In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
import copy
import random

2025-12-04 23:51:13.727103: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-04 23:51:13.727144: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-04 23:51:13.728484: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-04 23:51:13.736759: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-04 23:51:14.728131: W tensorflow/compiler/tf2

In [2]:
batch_size = 1024
learning_rate = 0.001

In [3]:
@tf.keras.saving.register_keras_serializable()
class MLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(units=128, activation=tf.nn.leaky_relu)
        self.dense2 = tf.keras.layers.Dense(units=1024, activation=tf.nn.leaky_relu)
        self.dense3 = tf.keras.layers.Dense(units=128, activation=tf.nn.leaky_relu)
        self.dense4 = tf.keras.layers.Dense(units=1024, activation=tf.nn.leaky_relu)
        self.dense5 = tf.keras.layers.Dense(units=8)

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        x = self.dense3(x)
        x = self.dense4(x)
        output = self.dense5(x)
        return output

In [4]:
class ParaServer:
    def __init__(self):
        self.model = MLP()
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    def upload(self, grads):
        self.optimizer.apply_gradients(grads_and_vars=zip(grads, self.model.variables))
        return self.model
    def download(self):
        return self.model
    def initModel(self, x):
        self.model(x)

In [5]:
def valiAll(index_epoch):
    m = ps.download()
    model = copy.deepcopy(m)
    y_v_p = model(X_v)
    va_mse = tf.reduce_mean(tf.square(y_v_p - y_v))
    va_rmse = tf.sqrt(va_mse)
    va_mae = tf.reduce_mean(tf.abs(y_v_p - y_v))
    va_r2 = 1 - tf.reduce_sum(tf.square(y_v_p - y_v)) / tf.reduce_sum(tf.square(y_v - tf.reduce_mean(y_v)))
    print("mse:{} rmse:{} mae:{} r2:{}".format(va_mse, va_rmse, va_mae, va_r2))
    r2sv[index_epoch] = va_r2.numpy()

In [6]:
class Node:
    def __init__(self, id, freq, mu=1e-4):
        self.id = id
        self.freq = freq
        self.model = MLP()
        self.mu = mu
        self.dataset1 = pd.read_csv('./20-24Trainset.csv', encoding='utf-8')
        self.dataset2 = pd.read_csv('./50-54Trainset.csv', encoding='utf-8')
        self.dataset = pd.concat([self.dataset1, self.dataset2], axis=0).sample(frac=1).reset_index(drop=True)
        self.dataset = self.dataset[self.dataset['freq'].isin(self.freq)]
        self.X = self.dataset.loc[:,'freq':'L2'].to_numpy(dtype = np.float32)
        self.y = self.dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)
        self.dataset_train = tf.data.Dataset.from_tensor_slices((self.X, self.y))
        self.dataset_train = self.dataset_train.shuffle(buffer_size=self.X.shape[0])
        self.dataset_train = self.dataset_train.batch(batch_size)
        self.dataset_train = self.dataset_train.prefetch(tf.data.experimental.AUTOTUNE)
    def train(self, index_epoch):
        m = ps.download()
        self.model = copy.deepcopy(m)
        global_weights = [tf.identity(w) for w in self.model.trainable_variables]
        for X, y in self.dataset_train:
            with tf.GradientTape() as tape:
                y_pred = self.model(X)
                tr_mse = tf.reduce_mean(tf.square(y_pred - y))
                prox_term = tf.add_n([
                        tf.nn.l2_loss(w - w0)
                        for w, w0 in zip(self.model.trainable_variables, global_weights)
                    ])
                loss = tr_mse + self.mu * prox_term
            tr_rmse = tf.sqrt(tr_mse)
            tr_mae = tf.reduce_mean(tf.abs(y_pred - y))
            tr_r2 = 1 - tf.reduce_sum(tf.square(y_pred - y)) / tf.reduce_sum(tf.square(y - tf.reduce_mean(y)))
            grads = tape.gradient(loss, self.model.variables)
            m = ps.upload(grads)
            self.model = copy.deepcopy(m)
        # if epoch_index in np.arange(0, num_epochs, 25).tolist() or epoch_index == num_epochs - 1:
        print("node:{} epoch:{}".format(self.freq, index_epoch))
        print("train mse:{} rmse:{} mae:{} r2:{}".format(tr_mse, tr_rmse, tr_mae, tr_r2))
        r2s[self.id][index_epoch] = tr_r2.numpy()

In [7]:
r2s = {0:{}, 1:{}, 2:{}, 3:{}, 4:{}}
r2sv = {}

In [8]:
test_dataset = pd.read_csv("testset.csv", encoding='utf-8').sample(frac=1).reset_index(drop=True)
X_v = test_dataset.loc[:,'freq':'L2'].to_numpy(dtype = np.float32)
y_v = test_dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)

In [9]:
ps = ParaServer()
ps.initModel(X_v)

2025-12-04 23:51:22.540565: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21505 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:02:00.0, compute capability: 8.9


In [10]:
nodeList = [Node(0, [2.0, 5.0]), Node(1, [2.1, 5.1]), Node(2, [2.2, 5.2]), Node(3, [2.3, 5.3]), Node(4, [2.4, 5.4])]

In [11]:
orders = [0, 1, 2, 3, 4]
turn = [np.array([[92, 146], [158, 255], [347, 475], [531, 555]]), 
        np.array([[42, 116], [226, 277], [363, 423], [543, 600]]), 
        np.array([[0, 200], [214, 252], [271, 347], [474, 528]]),
        np.array([[68, 132], [173, 305], [418, 423], [563, 587]]),
        np.array([[7, 151], [216, 305], [357, 360], [420, 538]]),]
for i in range(600):
    random.shuffle(orders)
    for j in orders:
        for l, r in turn[j]:
            if l <= i < r:
                nodeList[j].train(i)
    valiAll(i)

2025-12-04 23:51:29.481928: I external/local_xla/xla/service/service.cc:168] XLA service 0xfbde510 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-12-04 23:51:29.481961: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2025-12-04 23:51:29.488720: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-12-04 23:51:29.516010: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
I0000 00:00:1764892289.662527  166439 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


node:[2.2, 5.2] epoch:0
train mse:0.07971741259098053 rmse:0.28234273195266724 mae:0.22435137629508972 r2:0.3450627326965332
mse:0.07456520944833755 rmse:0.27306631207466125 mae:0.21348464488983154 r2:0.38928937911987305
node:[2.2, 5.2] epoch:1
train mse:0.06011894345283508 rmse:0.24519164860248566 mae:0.19243064522743225 r2:0.5071533918380737
mse:0.06733381003141403 rmse:0.2594875991344452 mae:0.2039984166622162 r2:0.4485166072845459
node:[2.2, 5.2] epoch:2
train mse:0.054559629410505295 rmse:0.23358002305030823 mae:0.18483982980251312 r2:0.5537854433059692
mse:0.05869413912296295 rmse:0.24226872622966766 mae:0.18986216187477112 r2:0.5192780494689941
node:[2.2, 5.2] epoch:3
train mse:0.04706648737192154 rmse:0.21694812178611755 mae:0.16906413435935974 r2:0.6152697801589966
mse:0.05605392903089523 rmse:0.23675711452960968 mae:0.18469592928886414 r2:0.5409021377563477
node:[2.2, 5.2] epoch:4
train mse:0.05063362047076225 rmse:0.2250191569328308 mae:0.17375530302524567 r2:0.5850770473480

In [13]:
for k, v in r2sv.items():
    print(v)

0.38928938
0.4485166
0.51927805
0.54090214
0.5276848
0.5580937
0.5998268
0.63192546
0.6424866
0.6459919
0.47740608
0.66539586
0.67435575
0.6618608
0.4936552
0.67333066
0.6819259
0.665478
0.5238187
0.50780094
0.689592
0.6923274
0.4972086
0.49439573
0.49128932
0.49398392
0.6951867
0.7010646
0.503155
0.69737124
0.70045865
0.4915883
0.5061728
0.48017716
0.5070158
0.70468146
0.49197203
0.49578673
0.50236464
0.6951562
0.7036005
0.7064851
0.62880325
0.5011764
0.70444286
0.7084304
0.6301499
0.6395643
0.49232507
0.7042168
0.7074432
0.6255511
0.6212164
0.6355235
0.7086147
0.48779738
0.6332582
0.7065704
0.62758714
0.7091212
0.48023093
0.7100881
0.6353573
0.6291164
0.47386092
0.6341263
0.4765613
0.6318542
0.62890524
0.6338696
0.6328166
0.64746004
0.6344051
0.4738531
0.47674322
0.6371038
0.70843565
0.6511383
0.47222286
0.6477045
0.70774835
0.647879
0.6292236
0.47614992
0.47999096
0.6300684
0.4616937
0.4714405
0.707971
0.7090962
0.6468729
0.70787084
0.4699121
0.6362174
0.6535114
0.7099079
0.47764653